# Data-poisoning a **neural-network** victim

The victim is a small MLP trained by online mini-batch SGD on the squared loss

$$f(x;\theta) = W_2\,\tanh(W_1x + b_1) + b_2,\qquad \ell_i = \tfrac12\big(f(x_i;\theta)-y_i\big)^2$$

where $\theta$ is the **flattened parameter vector**. With `d=5` and `victim_hidden=8`
that is $D = 8\cdot5 + 8 + 8 + 1 = 57$ parameters.

**Action-linearity still holds — and this is the whole point.** One SGD step on the
reweighted batch loss is

$$\theta_{t+1} = \theta_t - \tfrac{\eta}{m}\sum_i a_i\nabla_\theta\ell_i(\theta_t)
= \theta_t + \sum_i a_i\varphi_i$$

for **any** per-point loss, however nonlinear the model. The batch gradient is a *sum over
points*, so deleting/duplicating a point adds or removes its term exactly. The
decomposition never needed the victim to be linear — only its *update rule* to be.
(§1 verifies this numerically.)

What does change:

| | linear / logistic | MLP |
|---|---|---|
| `θ` lives in | data space, $\mathbb{R}^d$ | **parameter space**, $\mathbb{R}^{57}$ |
| `phi_batch` | closed form `resid·x` | **per-sample autograd** (`torch.func.vmap(grad)`) |
| start `θ₀` | zeros | small random init (zeros are a degenerate fixed point) |
| stability | robust | SGD **diverges** for `eta ≳ 1` — see §5 |

In [ ]:
import glob
import json
import os
import time
from dataclasses import replace

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

import sys
sys.path.insert(0, "src")

from config import EnvConfig, MethodConfig, TIER_NAME
from env import Environment, is_feasible
from fvi import train_fvi, evaluate, rollout
from agents import NullAgent, RandomAgent, GreedyAgent, MethodAgent
from decomposition import scores_and_interaction

# NB: do not `import viz` / `import diagnostics` -- they call matplotlib.use("Agg")
# at import time, which disables inline plots.
plt.rcParams["figure.dpi"] = 110
plt.rcParams.update({
    "savefig.dpi": 300,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial"],
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.fontsize": 9.5,
    "legend.frameon": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": "0.85",
    "grid.linewidth": 0.6,
    "grid.alpha": 0.6,
})
print("torch", torch.__version__)

## Configuration

`eta=0.3` sits inside the stable window (§5); above `eta≈0.5` the victim's own SGD
diverges and cost explodes. `victim_hidden` is kept moderate — `D` is the value
network's input dimension, so widening it further trades runtime for expressiveness.

Runtime ≈ 1–2 min for the main experiment (per-sample gradients make `phi_batch` costlier
than the closed-form tasks).

In [ ]:
ECFG = EnvConfig(d=5, m=10, T=10, eta=0.3, noise=0.1,
                 task="mlp", victim_hidden=8, init_scale=0.3)
CFG  = MethodConfig(select_mode="sort", c_edit=0.02, weight_decay=1e-3,
                    K_iters=50, rank_r=2)
EVAL_EPISODES, EVAL_SEED = 200, 999

INCLUDE_MODEL_FREE = True   # train BDQ / TRPO too (adds several minutes)
TRPO_ITERS = 50             # TRPO converges later than the others -- see note

torch.manual_seed(0)
env = Environment(ECFG, CFG)
print(f"task={ECFG.task}  data dim d={ECFG.d}  PARAMETER dim D={env.d}  "
      f"hidden={ECFG.victim_hidden}")
print(f"m={env.m}  T={env.T}  eta={ECFG.eta}  c_edit={CFG.c_edit}")

## 1. Verify action-linearity on the MLP

The claim the whole method rests on. We compare `theta + a @ Phi` against an explicit SGD
step on the `a`-reweighted batch loss, computed by autograd through the network.

In [ ]:
rng = torch.Generator().manual_seed(1)
z = env.stream.sample_batch(env.m, rng)
th = env.theta_0.clone()
a = torch.tensor([0., 1., 2., 1., 0., 2., 1., 1., 2., 0.])

lin = th + a @ env.victim.phi_batch(th, z)              # action-linear form
thg = th.detach().clone().requires_grad_(True)
L = (a * 0.5 * (env.victim.forward(thg, z[0]) - z[1]) ** 2).sum() / env.m
grad, = torch.autograd.grad(L, thg)
direct = th - ECFG.eta * grad                            # explicit SGD step

print(f"max |action-linear update  -  explicit SGD step| = "
      f"{float((lin - direct).abs().max()):.3e}   (float32 eps ≈ 1e-7)")
assert float((lin - direct).abs().max()) < 1e-5, "action-linearity broken!"

## 2. Train and compare

This cell now reads pre-computed results from `output/mlp_*.json` (produced by `run.py
--task mlp` from the terminal) instead of training here, matching
`experiment_linear.ipynb`/`experiment_logistic.ipynb`'s design -- multi-seed mean ± std
bars instead of one live seed's numbers. Produce them first:

```
./run_seeds.sh --task nn --seeds 10 --mode sort --bdq --trpo
./run_seeds.sh --task nn --seeds 10 --mode lowrank
./run_seeds.sh --task nn --seeds 10 --mode exact
```

Loading is precise the same way as the other two notebooks: an explicit `BASELINE` config
(matching `run.py`'s `TASK_DEFAULTS["mlp"]`, the same values the `ECFG`/`CFG` cell above
already uses) must match **exactly**, and any file carrying a `sens_axis` tag (from
`sensitivity/*.sh`) is excluded, so a one-off sensitivity point can't be mistaken for the
headline run.

**The "editing aggressiveness" panel from the old plot is gone.** Edits/step needs a live
rollout, which `results.json` doesn't save (same limitation `experiment_linear.ipynb`'s
closing note documents). §4 below still trains one live model for that kind of analysis,
if you need it -- this cell no longer does, since it's reading pre-computed numbers.

In [ ]:
TASK = "mlp"        # matches run.py's --task for this notebook
OUTDIR = "output"

# Mirrors run.py's TASK_DEFAULTS["mlp"] + the --rank/--k-max argparse defaults -- the
# same config the ECFG/CFG cell above already uses. "lowrank" is kept here (unlike
# experiment_linear.ipynb, which drops it) because §4 notes B is generically FULL rank
# for this task (D=57 > m=10), so low-rank isn't degenerate with first-order the way it
# is in the d < m linear case. "exact" (the un-approximated BQP optimum) is included
# too, as in experiment_linear.ipynb -- it shows how much low-rank/first-order leave
# on the table relative to solving the full interaction B exactly.
BASELINE = {"d": 5, "m": 10, "T": 10, "eta": 0.3, "noise": 0.1, "victim_hidden": 8,
            "init_scale": 0.3, "c_edit": 0.02, "weight_decay": 1e-3, "dual_iters": 20,
            "K_iters": 50, "rank_r": 2, "k_max": None}
MODES = ["sort", "lowrank", "exact"]


def cfg_of(run):
    cfg = {**run["env_config"], **run["method_config"]}
    return {f: cfg.get(f) for f in BASELINE}


def matches_baseline(cfg):
    return all(cfg[f] == v for f, v in BASELINE.items())


pattern = os.path.join(OUTDIR, f"{TASK}_*_results_*.json")
all_paths = glob.glob(pattern)
if not all_paths:
    raise SystemExit(
        f"no aggregated results found matching {pattern!r} -- run e.g.\n"
        f"  ./run_seeds.sh --task nn --seeds 10 --mode sort --bdq --trpo\n"
        f"  ./run_seeds.sh --task nn --seeds 10 --mode lowrank\n"
        f"  ./run_seeds.sh --task nn --seeds 10 --mode exact\n"
        f"first (run_seeds.sh calls aggregate.py automatically once all seeds finish)")

files, loaded, ambiguous = {}, {}, {}
for mode in MODES:
    candidates = []
    for path in all_paths:
        data = json.load(open(path))
        if data["mode"] != mode or data.get("sens_axis") is not None or not data["runs"]:
            continue
        if matches_baseline(cfg_of(data["runs"][0])):
            candidates.append((path, data))
    if not candidates:
        continue
    # exact-config matches can still differ in *what they trained*: prefer more seeds,
    # then more learners recorded, then newest path as a final tiebreak.
    def candidate_key(pd):
        _, d = pd
        return (len(d["runs"]), len(d["runs"][0]["costs"]), pd[0])
    path, data = max(candidates, key=candidate_key)
    if len(candidates) > 1:
        ambiguous[mode] = [os.path.basename(p) for p, _ in candidates]
    files[mode], loaded[mode] = path, data

if not loaded:
    raise SystemExit(
        f"no file in {OUTDIR}/ matches BASELINE={BASELINE} for any of {MODES} -- run e.g.\n"
        f"  ./run_seeds.sh --task nn --seeds 10 --mode sort --bdq --trpo\n"
        f"first, or edit BASELINE above to match the run you actually produced.")
missing = [m for m in MODES if m not in loaded]

merged = {}
for mode, data in loaded.items():
    for run in data["runs"]:
        s = run["seed"]
        dst = merged.setdefault(s, {"costs": {}, "history": {}, "train_times": {},
                                     "decision_times": {},
                                     "env_config": run["env_config"],
                                     "method_config": run["method_config"]})
        for key in ("costs", "history", "train_times", "decision_times"):
            for name, val in run.get(key, {}).items():
                dst[key].setdefault(name, val)

SEEDS_HEAD = sorted(merged)
runs_head = [merged[s] for s in SEEDS_HEAD]

PREFERRED_ORDER = ["null", "random", "greedy", "TRPO", "BDQ", "first-order", "low-rank", "exact"]
present = {n for r in runs_head for n in r["costs"]}
NAMES = [n for n in PREFERRED_ORDER if n in present] + sorted(present - set(PREFERRED_ORDER))

HEAD_COLORS = {"null": "#5b5b5b", "random": "#008300", "greedy": "#eb6834",
               "TRPO": "#4a3aa7", "BDQ": "#e34948", "first-order": "#2a78d6",
               "low-rank": "#1baf7a", "exact": "#12877f"}

print("loaded:", {m: os.path.basename(p) for m, p in files.items()})
if missing:
    print(f"no BASELINE-matching file for: {missing} (left out of the plot below)")
if ambiguous:
    for mode, names in ambiguous.items():
        print(f"[{mode}] {len(names)} files match BASELINE -- picked the one with the most "
              f"seeds (ties broken by most learners recorded, then newest): {names}")

cost_mean = {n: float(np.mean([r["costs"][n] for r in runs_head if n in r["costs"]])) for n in NAMES}
cost_std = {n: float(np.std([r["costs"][n] for r in runs_head if n in r["costs"]])) for n in NAMES}

D = BASELINE["victim_hidden"] * (BASELINE["d"] + 2) + 1   # MLPVictim's flattened param count

fig, ax = plt.subplots(figsize=(max(8, 1.1 * len(NAMES)), 4.4))
ax.bar(range(len(NAMES)), [cost_mean[k] for k in NAMES],
       yerr=[cost_std[k] for k in NAMES], capsize=5,
       color=[HEAD_COLORS.get(k, "C7") for k in NAMES])
if "null" in cost_mean:
    ax.axhline(cost_mean["null"], color=HEAD_COLORS["null"], ls="--", lw=1)
ax.set_xticks(range(len(NAMES))); ax.set_xticklabels(NAMES, rotation=15, fontsize=9)
ax.set_ylabel("mean episode cost")
ax.set_title(f"MLP victim (D={D} params, m={BASELINE['m']}, T={BASELINE['T']}, "
             f"{len(SEEDS_HEAD)} seeds)")
plt.tight_layout()
plt.show()

if "null" in cost_mean:
    nul = cost_mean["null"]
    for n in NAMES:
        print(f"  {n:18} cost={cost_mean[n]:9.3f} +/- {cost_std[n]:.3f}  "
              f"({cost_mean[n] / nul:5.3f} x null)")
print("\n'editing aggressiveness' (edits/step) and 'gap' aren't available from "
      "results.json -- see the markdown above.")

tiers_present = [t for t in ("first-order", "low-rank", "exact") if t in cost_mean]
if len(tiers_present) > 1:
    base = cost_mean[tiers_present[0]]
    for t in tiers_present[1:]:
        d = base - cost_mean[t]
        print(f"\n{tiers_present[0]} vs {t}: {d:+.3f} ({d / base * 100:+.2f}%)")

LEARNERS_HEAD = [n for n in ["first-order", "low-rank", "exact"] if n in present]
if LEARNERS_HEAD:
    FVI_LINESTYLE = {"first-order": "-", "low-rank": "--", "exact": ":"}
    fig, ax = plt.subplots(figsize=(7.5, 4.3))
    for name in LEARNERS_HEAD:
        H = np.array([r["history"][name] for r in runs_head if name in r["history"]])
        mu, sd = H.mean(0), H.std(0)
        it = np.arange(1, len(mu) + 1)
        ax.plot(it, mu, marker="o", linestyle=FVI_LINESTYLE.get(name, "-"),
                color=HEAD_COLORS[name], lw=2, label=f"{name} (FVI)",
                markevery=max(1, len(it) // 8))
        ax.fill_between(it, mu - sd, mu + sd, color=HEAD_COLORS[name], alpha=0.15)
    if "null" in cost_mean:
        ax.axhline(cost_mean["null"], color=HEAD_COLORS["null"], ls="--", lw=2, label="null")
    if "greedy" in cost_mean:
        ax.axhline(cost_mean["greedy"], color=HEAD_COLORS["greedy"], ls="-.", lw=2, label="greedy")
    ax.set_xlabel("FVI iteration"); ax.set_ylabel("held-out mean episode cost")
    ax.set_title("Learning curve (shaded = +/-1 std over seeds)")
    ax.legend(fontsize=9)
    plt.show()


### Train one live model (used by §4 below)

`results.json` doesn't save model weights or environments, so §4's per-state interaction
diagnostics -- which need the trained `V̂` and live `(α, B)` traces -- can't come
from the cell above. This retrains one seed at the same `BASELINE` config purely to supply
`env`/`V`/`res` to §4; it's not a substitute for §2's multi-seed numbers, so BDQ/TRPO
(already covered there) aren't retrained here.

In [ ]:
def compare(env, cfg, V, seeds_rng=1, tiers=("sort",)):
    """Evaluate every policy on the SAME held-out episodes."""
    agents = {
        "null":   NullAgent(),
        "random": RandomAgent(torch.Generator().manual_seed(2), cfg.balance, cfg.k_max),
        "greedy": GreedyAgent(env.victim, env.theta_target, cfg, mode="sort"),
    }
    for t in tiers:
        agents[TIER_NAME[t]] = MethodAgent(
            V, env.victim, env.theta_target, replace(cfg, select_mode=t),
            torch.Generator().manual_seed(seeds_rng))
    out = {}
    for n, ag in agents.items():
        c = evaluate(env, ag, torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
        e, g = behaviour(env, ag)
        out[n] = {"cost": c, "edits": e, "gap": g, "agent": ag}
    return out


def behaviour(env, agent, n_episodes=60):
    """Mean edits/step and mean ||theta_t - theta_target||; also asserts feasibility."""
    rng = torch.Generator().manual_seed(EVAL_SEED)
    edits = gap = 0.0
    steps = 0
    for _ in range(n_episodes):
        theta = env.theta_0.clone()
        for t in range(env.T):
            z = env.stream.sample_batch(env.m, rng)
            a = agent.act(theta, z, t)
            assert is_feasible(a, CFG.balance, CFG.k_max), "infeasible action!"
            edits += float(np.abs(a.numpy() - 1).sum())
            theta = env.victim.update(theta, z, a).detach()
            gap += float((theta - env.theta_target).norm())
            steps += 1
    return edits / steps, gap / steps


t0 = time.perf_counter()
V, hist = train_fvi(env, CFG, seed=0, verbose=False, return_history=True,
                    best_checkpoint=True, eval_episodes=EVAL_EPISODES, eval_seed=EVAL_SEED)
print(f"trained in {time.perf_counter()-t0:.0f}s")

res = compare(env, CFG, V)
nul = res["null"]["cost"]
for name, r in res.items():
    print(f"  {name:12} cost={r['cost']:9.3f}  ({r['cost']/nul:5.3f} x null)"
          f"  edits={r['edits']:5.2f}  gap={r['gap']:7.3f}")
print("\nsee §2 above for the definitive multi-seed headline numbers -- this is one "
      "seed's live model, kept only to supply agents/env/V to §4.")


## 3. The learned function under each attack policy (d = 1)

In 1-D the victim's entire hypothesis is a curve, so the attack is directly visible. A
separate `d=1` run (D = 25 parameters) with **every policy on the identical data stream**.

**Reading the panels.** Marker shape is the attacker's action (● kept, ✕ deleted,
▲ duplicated). Four curves appear in every panel:

| curve | meaning |
|---|---|
| grey dotted | the teacher `y = sin(θ*x)` that generated the data |
| **green dashed** | `f(·; θ̃)` — the function the attacker's *target weights* encode |
| blue thin | clean SGD (null) — where the victim lands with no attack |
| **black thick** | `f(·; θ_T)` — where *this* policy drove the victim |

A successful attack pulls the black curve off the blue one and toward the green one. Two
numbers per panel: `‖θ_T − θ̃‖` is the quantity the cost actually minimises, while
**RMSE to the target function** measures the same thing in *function* space — worth
reporting separately, because weight-space symmetry means two different `θ` can encode
nearly the same `f`.

In [ ]:
E1, C1 = replace(ECFG, d=1), replace(CFG, K_iters=20)
torch.manual_seed(0)
env1 = Environment(E1, C1)
print(f"d=1 MLP victim: D = {env1.d} parameters")

V1 = train_fvi(env1, C1, seed=0, verbose=False, best_checkpoint=True,
               eval_episodes=100, eval_seed=EVAL_SEED)

policies = {
    "null (clean SGD)": NullAgent(),
    "random": RandomAgent(torch.Generator().manual_seed(2), C1.balance, C1.k_max),
    "greedy": GreedyAgent(env1.victim, env1.theta_target, C1, mode="sort"),
}
if INCLUDE_MODEL_FREE:            # BDQ dominates the runtime here (per-sample autograd)
    from bdq import train_bdq, BDQAgent
    from trpo import train_trpo, TRPOAgent
    torch.manual_seed(0)
    P1 = train_trpo(env1, replace(C1, K_iters=TRPO_ITERS), seed=0, verbose=False,
                    best_checkpoint=True, eval_episodes=100, eval_seed=EVAL_SEED)
    torch.manual_seed(0)
    Q1 = train_bdq(env1, C1, seed=0, verbose=False, best_checkpoint=True,
                   eval_episodes=100, eval_seed=EVAL_SEED)
    policies["TRPO"] = TRPOAgent(P1, env1, C1, torch.Generator().manual_seed(1))
    policies["BDQ"] = BDQAgent(Q1, env1, C1, torch.Generator().manual_seed(1))
policies[TIER_NAME[C1.select_mode]] = MethodAgent(V1, env1.victim, env1.theta_target, C1,
                                        torch.Generator().manual_seed(1))

rng = torch.Generator().manual_seed(5)
batches = [env1.stream.sample_batch(env1.m, rng) for _ in range(env1.T)]   # shared stream


def run_on(agent):
    th = env1.theta_0.clone()
    xs, ys, acts = [], [], []
    for t, z in enumerate(batches):
        a = agent.act(th, z, t)
        xs.append(z[0].reshape(-1)); ys.append(z[1]); acts.append(a)
        th = env1.victim.update(th, z, a).detach()
    return th, torch.cat(xs).numpy(), torch.cat(ys).numpy(), torch.cat(acts).numpy()


X_all = torch.cat([z[0].reshape(-1) for z in batches]).numpy()
xs_grid = torch.linspace(float(X_all.min()), float(X_all.max()), 300)[:, None]
f = lambda th: env1.victim.forward(th, xs_grid).detach().numpy()
f_star = torch.sin(xs_grid.reshape(-1) * env1.stream.theta_star).numpy()   # non-linear teacher
f_targ = f(env1.theta_target)
f_null = f(run_on(NullAgent())[0])

ncols = 3
nrows = (len(policies) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.5 * nrows),
                         sharex=True, sharey=True, squeeze=False)
rows = []
for ax, (name, ag) in zip(axes.flat, policies.items()):
    th, X, Y, A = run_on(ag)
    keep, dele, dup = A == 1, A == 0, A == 2
    ax.scatter(X[keep], Y[keep], s=13, c="0.55", alpha=.45, zorder=2)
    ax.scatter(X[dele], Y[dele], s=75, c="#b3202c", marker="x", linewidths=2.3, zorder=4)
    ax.scatter(X[dup], Y[dup], s=62, c="#1f9c4e", marker="^",
               edgecolors="k", linewidths=.8, zorder=4)
    ax.plot(xs_grid.numpy(), f_star, ":", color="0.35", lw=1.8, zorder=3)
    ax.plot(xs_grid.numpy(), f_targ, "--", color="#00b050", lw=2.4, zorder=5)
    ax.plot(xs_grid.numpy(), f_null, "-", color="tab:blue", lw=1.6, alpha=.75, zorder=5)
    ax.plot(xs_grid.numpy(), f(th), "-", color="k", lw=3.0, zorder=6)
    gap = float((th - env1.theta_target).norm())
    fdist = float(np.sqrt(((f(th) - f_targ) ** 2).mean()))
    rows.append((name, gap, fdist, int(dele.sum()), int(dup.sum())))
    ax.set_title(f"{name}\n‖θ_T − θ̃‖ = {gap:.2f}   RMSE to target fn = {fdist:.2f}\n"
                 f"{int(dele.sum())} deleted / {int(dup.sum())} duplicated", fontsize=9.5)
    ax.set_xlabel("x"); ax.set_ylabel("y / f(x)")
for ax in list(axes.flat)[len(policies):]:
    ax.axis("off")

from matplotlib.lines import Line2D
fig.legend(handles=[
    Line2D([0], [0], color="0.35", ls=":", lw=1.8, label="teacher  y = sin(θ*x)"),
    Line2D([0], [0], color="#00b050", ls="--", lw=2.4, label="target  f(·; θ̃)"),
    Line2D([0], [0], color="tab:blue", lw=1.6, label="clean SGD (null)"),
    Line2D([0], [0], color="k", lw=3.0, label="this policy's f(·; θ_T)"),
    Line2D([0], [0], marker="o", color="0.55", ls="None", label="kept"),
    Line2D([0], [0], marker="x", color="#b3202c", ls="None", label="deleted"),
    Line2D([0], [0], marker="^", color="#1f9c4e", mec="k", ls="None", label="duplicated")],
    loc="lower center", ncol=7, fontsize=9)
fig.suptitle(f"MLP victim (d=1, D={env1.d}): the learned function under each attack "
             f"policy — identical data stream", fontsize=13)
fig.tight_layout(rect=(0, 0.07, 1, 0.95)); plt.show()

print(f"{'policy':20}{'||θ_T-θ̃||':>12}{'RMSE to target fn':>19}{'del':>6}{'dup':>6}")
for r in rows:
    print(f"{r[0]:20}{r[1]:>12.3f}{r[2]:>19.3f}{r[3]:>6d}{r[4]:>6d}")
print("\nSingle episode, single seed — illustrative; the quantitative comparison is §2.")

In [ ]:
rows = []
for ax, (name, ag) in zip(axes.flat, policies.items()):
    th, X, Y, A = run_on(ag)
    keep, dele, dup = A == 1, A == 0, A == 2
    ax.scatter(X[keep], Y[keep], s=13, c="0.55", alpha=.45, zorder=2)
    ax.scatter(X[dele], Y[dele], s=75, c="#b3202c", marker="x", linewidths=2.3, zorder=4)
    ax.scatter(X[dup], Y[dup], s=62, c="#1f9c4e", marker="^",
               edgecolors="k", linewidths=.8, zorder=4)
    ax.plot(xs_grid.numpy(), f_star, ":", color="0.35", lw=1.8, zorder=3)
    ax.plot(xs_grid.numpy(), f_targ, "--", color="#00b050", lw=2.4, zorder=5)
    ax.plot(xs_grid.numpy(), f_null, "-", color="tab:blue", lw=1.6, alpha=.75, zorder=5)
    ax.plot(xs_grid.numpy(), f(th), "-", color="k", lw=3.0, zorder=6)
    gap = float((th - env1.theta_target).norm())
    fdist = float(np.sqrt(((f(th) - f_targ) ** 2).mean()))
    rows.append((name, gap, fdist, int(dele.sum()), int(dup.sum())))
    ax.set_title(f"{name}\n‖θ_T − θ̃‖ = {gap:.2f}   RMSE to target fn = {fdist:.2f}\n"
                 f"{int(dele.sum())} deleted / {int(dup.sum())} duplicated", fontsize=9.5)
    ax.set_xlabel("x"); ax.set_ylabel("y / f(x)")
for ax in list(axes.flat)[len(policies):]:
    ax.axis("off")

from matplotlib.lines import Line2D
fig.legend(handles=[
    Line2D([0], [0], color="0.35", ls=":", lw=1.8, label="teacher  y = sin(θ*x)"),
    Line2D([0], [0], color="#00b050", ls="--", lw=2.4, label="target  f(·; θ̃)"),
    Line2D([0], [0], color="tab:blue", lw=1.6, label="clean SGD (null)"),
    Line2D([0], [0], color="k", lw=3.0, label="this policy's f(·; θ_T)"),
    Line2D([0], [0], marker="o", color="0.55", ls="None", label="kept"),
    Line2D([0], [0], marker="x", color="#b3202c", ls="None", label="deleted"),
    Line2D([0], [0], marker="^", color="#1f9c4e", mec="k", ls="None", label="duplicated")],
    loc="lower center", ncol=7, fontsize=9)
fig.suptitle(f"MLP victim (d=1, D={env1.d})", fontsize=13)
fig.tight_layout(rect=(0, 0.07, 1, 0.95)); 
plt.show()

## 4. Interaction diagnostics

Note `rank(B) ≤ min(m, D)`; with `D=57 > m=10` the interaction matrix is generically
**full rank**, unlike the `d < m` linear case where `m-d` eigenvalues vanish.

In [ ]:
alpha_rel, spec = [], []
rng = torch.Generator().manual_seed(EVAL_SEED)
theta = env.theta_0.clone()
for i in range(60):
    t = i % env.T
    if t == 0:
        theta = env.theta_0.clone()
    z = env.stream.sample_batch(env.m, rng)
    alpha, B = scores_and_interaction(V, theta, z, t, env.victim, env.theta_target,
                                      CFG.lam, need_B=True, use_hessian=CFG.use_hessian_in_B)
    a = res["first-order"]["agent"].act(theta, z, t)
    lin = float(a @ alpha)
    alpha_rel.append(abs(float(0.5 * a @ B @ a)) / (abs(lin) + 1e-12))
    spec.append(np.sort(np.abs(np.linalg.eigvalsh(0.5 * (B + B.T).numpy())))[::-1])
    theta = env.victim.update(theta, z, a).detach()

spec = np.array(spec)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.2))
a1.hist(alpha_rel, bins=25, color="C0", alpha=0.85)
a1.axvline(np.median(alpha_rel), color="k", ls="--", lw=1.5,
           label=f"median = {np.median(alpha_rel):.3f}")
a1.set_xlabel("|½ aᵀBa| / |aᵀα|"); a1.set_ylabel("states")
a1.set_title("Relative interaction (what `sort` discards)"); a1.legend(fontsize=9)

a2.semilogy(np.arange(1, spec.shape[1] + 1), spec.mean(0), "o-", color="C0", lw=2)
a2.set_xlabel("index k"); a2.set_ylabel("mean |eigenvalue| of B")
a2.set_title(f"Spectrum of B  (rank(B) ≤ min(m,d) = {min(ECFG.m, env.d)})")
plt.tight_layout(); plt.show()
print(f"relative interaction: median {np.median(alpha_rel):.4f}  mean {np.mean(alpha_rel):.4f}")

## 5. Stability: the victim's own SGD sets the usable `eta`

A nonlinear victim can diverge on its own. Past `eta ≈ 0.5` the MLP's training blows up and
every cost — including `null` — explodes, so nothing about the attack is measurable there.
This is a constraint the linear victim never imposed.

Also watch `greedy` at `eta=0.5`: it becomes **worse than doing nothing**. Myopic editing
destabilises a nonlinear victim, while the planning method still helps.

In [ ]:
print(f"{'eta':>5}{'null':>12}{'greedy':>12}{'first-order':>12}{'g/null':>8}{'m/null':>8}")
etas, ratios_m, ratios_g = [], [], []
for eta in [0.1, 0.2, 0.3, 0.5]:
    e = replace(ECFG, eta=eta)
    torch.manual_seed(0)
    en = Environment(e, CFG)
    Vs = train_fvi(en, CFG, seed=0, verbose=False, best_checkpoint=True,
                   eval_episodes=EVAL_EPISODES, eval_seed=EVAL_SEED)
    ag = MethodAgent(Vs, en.victim, en.theta_target, CFG, torch.Generator().manual_seed(1))
    gr = GreedyAgent(en.victim, en.theta_target, CFG, mode="sort")
    n_ = evaluate(en, NullAgent(), torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
    g_ = evaluate(en, gr, torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
    m_ = evaluate(en, ag, torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
    etas.append(eta); ratios_m.append(m_ / n_); ratios_g.append(g_ / n_)
    print(f"{eta:>5}{n_:>12.3f}{g_:>12.3f}{m_:>12.3f}{g_/n_:>8.3f}{m_/n_:>8.3f}")

plt.figure(figsize=(7.5, 4.2))
plt.plot(etas, ratios_m, "o-", color="#2a78d6", lw=2, label=f"{TIER_NAME[CFG.select_mode]} / null")
plt.plot(etas, ratios_g, "s-.", color="#eb6834", lw=2, label="greedy / null")
plt.axhline(1.0, color="0.7", ls=":", lw=1.5, label="no better than null")
plt.xlabel("victim learning rate eta"); plt.ylabel("cost / null (lower = stronger)")
plt.title("Attack strength vs victim stability"); plt.legend(fontsize=9); plt.show()

## 6. Sensitivity analysis  *(heavy — retrains every learner at every point)*

Everything above is one point in configuration space. This section sweeps the axes that
either **carry the claim** or **bound its scope**, with the same fairness protocol as §2:
every learner is retrained at every point, all policies are scored on the identical
held-out episodes, and feasibility is asserted throughout.

| § | axis | why it matters |
|---|---|---|
| 6a | `c_edit` | **the** axis — `c_edit` is what demands per-point credit resolution |
| 6b | `m` | separates action-space growth from edit economics |
| 6c | `T` | isolates what the learned cost-to-go buys over myopic greedy |
| 6d | linearity violation | the MLP's update is still *exactly* action-linear (§1) — this measures how much that breaks under clipping / multi-step SGD, same as it would for any victim |
| 6e | `rank_r` | how much of `B` is actually needed — here `D=57 > m=10`, so `B` is generically full rank (§4), unlike the `d<m` linear case |

Per-sample autograd makes every retrain here costlier than the linear/logistic notebook, so
this section runs a **single seed** throughout (matching the rest of this notebook) rather
than the multi-seed protocol used for the linear notebook's headline comparison.

**Run the Configuration cell and §2 first.** This section reuses `env`, `V`, `CFG`, `ECFG`,
`EVAL_EPISODES`/`EVAL_SEED`, `INCLUDE_MODEL_FREE`, `TRPO_ITERS` and the `behaviour` helper
defined right after Configuration.

**Not swept, deliberately:** `λ` is redundant with `c_edit`, for the same reason as the
linear notebook — the objective is `c_edit·Σ|aᵢ−1| + (λ/2)‖θ′−θ̃‖²`, so `(λ, c_edit) →
(cλ, c·c_edit)` multiplies `Q` by `c` and leaves the argmin unchanged.


In [ ]:
# --- shared sweep machinery -------------------------------------------------
from config import TIER_MODE
import os

RUN_CEDIT     = True    # 6a
RUN_M_SENS    = True    # 6b
RUN_T_SENS    = True    # 6c
RUN_VIOLATION = True    # 6d
RUN_RANK      = True    # 6e (free -- no retraining)

SEEDS = (0,)             # per-sample autograd makes this notebook single-seed throughout
K_COMPARE = CFG.K_iters
SWEEP_LEARNERS = ["first-order", "low-rank", "BDQ", "TRPO"] if INCLUDE_MODEL_FREE \
                 else ["first-order", "low-rank"]
SWEEP_FREE = ["null", "greedy"]

# Each axis point in a sweep trains + evaluates on its own Environment, so points are
# fully independent and can run in separate processes. Toggle here (applies to every
# `sweep(...)` call below) or per-call via `sweep(..., parallel=True)`. Needs the
# `multiprocess` package (dill-based) rather than stdlib multiprocessing, because
# `make_cfgs`/`post_env` are notebook-local lambdas/closures (6d's post_env even
# closes over notebook-local classes like ClipVictim) that plain pickle can't send
# to a worker -- dill can. `pip install multiprocess` if it's missing.
SWEEP_PARALLEL  = False
SWEEP_N_WORKERS = max(1, (os.cpu_count() or 2) - 1)

COLORS = {"null": "#5b5b5b", "greedy": "#eb6834", "TRPO": "#4a3aa7",
          "BDQ": "#e34948", "first-order": "#2a78d6",
          "low-rank": "#1baf7a", "exact": "#e87ba4"}
MARKERS = {"null": "D", "greedy": "s", "TRPO": "^", "BDQ": "X",
           "first-order": "o", "low-rank": "o", "exact": "o"}

if INCLUDE_MODEL_FREE:
    from bdq import train_bdq, BDQAgent
    from trpo import train_trpo, TRPOAgent


def build_agents(env, cfg, seed, learners=SWEEP_LEARNERS):
    """Train `learners` on this env/cfg; return {name: agent} incl. the free baselines."""
    ags = {"null": NullAgent(),
           "greedy": GreedyAgent(env.victim, env.theta_target,
                                 replace(cfg, select_mode="sort"), mode="sort")}
    for name in learners:
        torch.manual_seed(seed)                       # identical init across learners
        if name == "BDQ":
            net = train_bdq(env, cfg, seed=seed, verbose=False, best_checkpoint=True,
                            eval_episodes=EVAL_EPISODES, eval_seed=EVAL_SEED)
            ags[name] = BDQAgent(net, env, cfg, torch.Generator().manual_seed(seed + 1))
        elif name == "TRPO":
            cfg_t = replace(cfg, K_iters=max(cfg.K_iters, TRPO_ITERS))
            net = train_trpo(env, cfg_t, seed=seed, verbose=False, best_checkpoint=True,
                             eval_episodes=EVAL_EPISODES, eval_seed=EVAL_SEED)
            ags[name] = TRPOAgent(net, env, cfg_t, torch.Generator().manual_seed(seed + 1))
        else:
            cfg_t = replace(cfg, select_mode=TIER_MODE[name])
            V_ = train_fvi(env, cfg_t, seed=seed, verbose=False, best_checkpoint=True,
                           eval_episodes=EVAL_EPISODES, eval_seed=EVAL_SEED)
            ags[name] = MethodAgent(V_, env.victim, env.theta_target, cfg_t,
                                    torch.Generator().manual_seed(seed + 1))
    return ags


def _sweep_point(job):
    """Run every seed for one axis point; returns (i, v, per_c, per_e).

    Self-contained so it can be shipped to a worker process unchanged: only touches
    its own argument tuple plus names already at module/notebook scope (build_agents,
    Environment, evaluate, behaviour, SWEEP_FREE), which `multiprocess`/dill pulls
    along when it serializes the function.
    """
    i, v, ecfg_v, cfg_v, seeds, learners, post_env, n_beh = job
    names = SWEEP_FREE + list(learners)
    per_c = {n: [] for n in names}
    per_e = {n: [] for n in names}
    for s in seeds:
        torch.manual_seed(s)
        env_v = Environment(ecfg_v, cfg_v)
        if post_env is not None:
            post_env(env_v, v)
        ags = build_agents(env_v, cfg_v, s, learners)
        for n in names:
            per_c[n].append(evaluate(env_v, ags[n],
                            torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES))
            e_, _ = behaviour(env_v, ags[n], n_episodes=n_beh)
            per_e[n].append(e_)
    return i, v, per_c, per_e


def _sweep_worker_init():
    torch.set_num_threads(1)   # each worker gets its own process; don't oversubscribe BLAS


def sweep(values, make_cfgs, seeds=SEEDS, learners=SWEEP_LEARNERS, label="x",
          post_env=None, n_beh=40, parallel=None, n_workers=None):
    """1-D sweep: retrain every learner at every point, score on the shared episodes.

    make_cfgs(v) -> (EnvConfig, MethodConfig) for axis value v.
    post_env(env, v) may mutate the freshly built Environment (used by 6d).
    parallel=True (default: SWEEP_PARALLEL) runs the axis points in separate worker
    processes via `multiprocess.Pool` -- each point is an independent train+eval, so
    this parallelises cleanly across `n_workers` (default: SWEEP_N_WORKERS) cores.
    Sequential and parallel modes produce identical `res` (same per-point seeding);
    only wall-clock time and the order progress lines print in differ.
    Returns {name: {"cost", "cost_sd", "edits"}}, each a list over `values`.
    """
    parallel = SWEEP_PARALLEL if parallel is None else parallel
    names = SWEEP_FREE + list(learners)
    res = {n: {"cost": [], "cost_sd": [], "edits": []} for n in names}
    jobs = [(i, v, *make_cfgs(v), seeds, learners, post_env, n_beh)
            for i, v in enumerate(values)]

    if not parallel:
        points = [None] * len(jobs)
        for job in jobs:
            i, v, per_c, per_e = _sweep_point(job)
            points[i] = (v, per_c, per_e)
            print(f"  {label}={v}: " +
                  "  ".join(f"{n}={np.mean(per_c[n]):.2f}" for n in names), flush=True)
    else:
        try:
            import multiprocess as mp
        except ImportError as exc:
            raise ImportError(
                "parallel=True needs the `multiprocess` package (dill-based; handles "
                "notebook-local lambdas/classes that stdlib multiprocessing can't "
                "pickle) -- `pip install multiprocess`."
            ) from exc
        nw = min(n_workers or SWEEP_N_WORKERS, len(jobs))
        points = [None] * len(jobs)
        # spawn, not fork: torch autograd's worker threads (already running by
        # the time this cell executes) don't survive a fork -- see
        # https://github.com/pytorch/pytorch/wiki/Autograd-and-Fork
        with mp.get_context("spawn").Pool(nw, initializer=_sweep_worker_init) as pool:
            for i, v, per_c, per_e in pool.imap_unordered(_sweep_point, jobs):
                points[i] = (v, per_c, per_e)
                print(f"  {label}={v}: " +
                      "  ".join(f"{n}={np.mean(per_c[n]):.2f}" for n in names), flush=True)

    for v, per_c, per_e in points:
        for n in names:
            res[n]["cost"].append(float(np.mean(per_c[n])))
            res[n]["cost_sd"].append(float(np.std(per_c[n])))
            res[n]["edits"].append(float(np.mean(per_e[n])))
    return res


def plot_sweep(xs, res, xlabel, title, panel2="edits", xticks=None):
    """Left: cost/null. Right: edits/step ('edits') or planning premium ('premium')."""
    names = list(res)
    nul = np.array(res["null"]["cost"])
    fig, (p1, p2) = plt.subplots(1, 2, figsize=(12.5, 4.3))
    for n in names:
        p1.plot(xs, np.array(res[n]["cost"]) / nul, f"{MARKERS[n]}-", color=COLORS[n], lw=2, label=n)
    p1.axhline(1.0, color="0.7", ls=":", lw=1)
    p1.set_xlabel(xlabel); p1.set_ylabel("cost / null  (lower = stronger attack)")
    p1.set_title(title); p1.legend(fontsize=8)

    if panel2 == "premium":
        g = np.array(res["greedy"]["cost"])
        for n in ["first-order", "low-rank"]:
            if n in res:
                p2.plot(xs, (g - np.array(res[n]["cost"])) / g * 100, f"{MARKERS[n]}-",
                        color=COLORS[n], lw=2, label=f"{n} vs greedy")
        p2.axhline(0, color="k", lw=0.8)
        p2.set_ylabel("cost reduction vs greedy (%)")
        p2.set_title("Planning premium — what the learned V̂ buys")
    else:
        for n in names:
            if n != "null":
                p2.plot(xs, res[n]["edits"], f"{MARKERS[n]}-", color=COLORS[n], lw=2, label=n)
        p2.set_ylabel("mean points edited per step")
        p2.set_title("Willingness to pay for edits")
    p2.set_xlabel(xlabel); p2.legend(fontsize=8)
    for p in (p1, p2):
        if xticks is not None:
            p.set_xticks(xticks)
    plt.tight_layout(); plt.show()


def decision_latency_ms(env, agent, n=200, seed=1):
    rng = torch.Generator().manual_seed(seed)
    theta, z = env.theta_0.clone(), env.stream.sample_batch(env.m, rng)
    agent.act(theta, z, 0)                                   # warm-up
    t0 = time.perf_counter()
    for k in range(n):
        agent.act(theta, z, k % env.T)
    return (time.perf_counter() - t0) / n * 1e3


print("sweep helpers ready — flags:",
      {"6a": RUN_CEDIT, "6b": RUN_M_SENS, "6c": RUN_T_SENS,
       "6d": RUN_VIOLATION, "6e": RUN_RANK})

### 6a. Edit price `c_edit` — the credit-resolution axis

Same hypothesis as the linear notebook: the three baselines differ in how finely they can
attribute cost to individual points — TRPO gets **one scalar** advantage for the whole
joint action, BDQ gets **per-branch** Q-values, ours gets a **per-point analytic `αᵢ`**.
`c_edit` is exactly the knob that demands that discrimination, and the MLP's larger,
full-rank `B` (§4) only makes fine-grained credit assignment harder to substitute for.

`c_edit=0` is included as the anchor where the axis should *not* discriminate. Points are
scaled to the same ratios as the linear notebook's sweep, centered on this notebook's
baseline `c_edit=0.02` instead of `0.1`.


In [ ]:
CEDITS      = [0.08, 0.0, 0.005, 0.01, 0.02, 0.04]
CEDIT_SEEDS = SEEDS

if RUN_CEDIT:
    res_ce = sweep(CEDITS,
                   lambda v: (ECFG, replace(CFG, c_edit=v, K_iters=K_COMPARE)),
                   seeds=CEDIT_SEEDS, label="c_edit")
    plot_sweep(CEDITS, res_ce, "edit price  c_edit",
               f"Attack strength vs edit price ({len(CEDIT_SEEDS)} seeds)",
               xticks=CEDITS)

    # where does each policy give up? first c_edit at which it is within 5% of null
    print("\ncollapse point (first c_edit with cost/null > 0.95):")
    nul = np.array(res_ce["null"]["cost"])
    for n in SWEEP_FREE[1:] + SWEEP_LEARNERS:
        r = np.array(res_ce[n]["cost"]) / nul
        hit = [c for c, v in zip(CEDITS, r) if v > 0.95]
        print(f"  {n:12} {'c_edit = ' + str(hit[0]) if hit else 'survives the whole sweep':30}"
              f"  (cost/null at c_edit={CEDITS[-1]}: {r[-1]:.3f})")
else:
    print("skipped — set RUN_CEDIT = True")


### 6b. Batch size `m` — action-space growth vs edit economics

Two confounded effects grow with `m`, exactly as in the linear notebook:

1. the joint action space grows as `3^m` — the reason BDQ exists;
2. per-point influence shrinks as `φᵢ ∝ 1/m` (the victim averages the batch), so at
   **fixed** `c_edit` every edit becomes less profitable and *all* policies drift to null.

Holding `c_edit·m` constant (right panel) neutralises effect 2.

Read the two panels together: any degradation that survives on the **right** is a genuine
scaling limit; degradation that appears only on the **left** is the edit price rising in
disguise.


In [ ]:
MS_SENS = [4, 6, 8, 10, 14]

if RUN_M_SENS:
    res_m = {}
    for scaled in (False, True):
        tag = "c_edit*m held constant" if scaled else "c_edit fixed"
        print(f"[{tag}]")
        res_m[scaled] = sweep(
            MS_SENS,
            lambda m, sc=scaled: (replace(ECFG, m=m),
                                  replace(CFG, K_iters=K_COMPARE,
                                          c_edit=CFG.c_edit * ECFG.m / m if sc
                                          else CFG.c_edit)),
            seeds=(SEEDS[0],), label="m")

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3), sharey=True)
    for ax, scaled, ttl in [(axes[0], False, f"c_edit fixed at {CFG.c_edit}"),
                            (axes[1], True, f"c_edit·m held at {CFG.c_edit*ECFG.m:.1f}")]:
        r = res_m[scaled]
        nul = np.array(r["null"]["cost"])
        for n in r:
            ax.plot(MS_SENS, np.array(r[n]["cost"]) / nul, f"{MARKERS[n]}-",
                    color=COLORS[n], lw=2, label=n)
        ax.axhline(1.0, color="0.7", ls=":", lw=1)
        ax.set_xlabel("batch size m   (action space = 3^m)")
        ax.set_xticks(MS_SENS); ax.set_title(ttl); ax.legend(fontsize=8)
    axes[0].set_ylabel("cost / null")
    fig.suptitle("Scaling in m", y=1.02)
    plt.tight_layout(); plt.show()

    print(f"\n{'m':>4}{'3^m':>12}   " +
          "".join(f"{n:>14}" for n in SWEEP_LEARNERS))
    for i, m in enumerate(MS_SENS):
        vals = [res_m[True][n]["cost"][i] / res_m[True]["null"]["cost"][i]
                for n in SWEEP_LEARNERS]
        print(f"{m:>4}{3**m:>12,}   " + "".join(f"{v:>14.3f}" for v in vals) +
              "   (c_edit·m fixed)")
else:
    print("skipped — set RUN_M_SENS = True")


### 6c. Horizon `T` — what the learned cost-to-go buys

Identical logic to the linear notebook: `greedy` **is** the myopic special case of our own
decomposition (`V̂ ≡ 0`), run with the identical selector and `c_edit`, so the gap between
`first-order` and `greedy` isolates one thing: the value of the learned cost-to-go. §5
already showed myopia can be actively harmful on this nonlinear victim (`greedy` loses to
`null` at `eta=0.5`) — this sweep asks how the planning premium grows with horizon.

**Built-in sanity check.** At `T=1` there is no future — `V̂(θ, t+1)` is evaluated at
`t+1 = T`, which the value net defines as `0` — so `α` reduces exactly to greedy's and the
two policies must coincide. The cell asserts it.


In [ ]:
TS_SENS = [1, 2, 4, 6, 10, 14]

if RUN_T_SENS:
    res_T = sweep(TS_SENS,
                  lambda T: (replace(ECFG, T=T), replace(CFG, K_iters=K_COMPARE)),
                  seeds=(SEEDS[0],), label="T")
    plot_sweep(TS_SENS, res_T, "horizon T", "Attack strength vs planning horizon",
               panel2="premium", xticks=TS_SENS)

    g1, f1 = res_T["greedy"]["cost"][0], res_T["first-order"]["cost"][0]
    print(f"\nT=1 sanity check: greedy={g1:.4f}  first-order={f1:.4f}  "
          f"|diff|={abs(g1-f1):.2e}")
    assert abs(g1 - f1) < 1e-6 * max(1.0, abs(g1)), \
        "at T=1 the method must reduce to greedy (V̂(·,T)=0) -- check the t+1 indexing"

    g = np.array(res_T["greedy"]["cost"])
    prem = (g - np.array(res_T["first-order"]["cost"])) / g * 100
    print("planning premium vs greedy:  " +
          "  ".join(f"T={T}: {p:+.1f}%" for T, p in zip(TS_SENS, prem)))
else:
    print("skipped — set RUN_T_SENS = True")


### 6d. Graded violation of action-linearity — the scope of the method

§1 verified the MLP's update is *exactly* action-linear — the decomposition doesn't need
the victim to be linear, only its update rule to be. That exactness is still bought with a
modelling assumption: `θ_{t+1} = θ_t + Σ aᵢφᵢ`. This section reuses the linear notebook's
two knobs to convert that assumption into a measured bound, now on a genuinely nonlinear
victim:

| knob | what it breaks | exact at |
|---|---|---|
| **global grad-norm clip `τ`** | sum structure — the clip applies to `Σ aᵢ∇ℓᵢ` *after* aggregation, so the update is no longer separable | `τ = ∞` |
| **inner SGD steps `n`** | "exactly one step" — steps 2…`n` re-linearise at a `θ` the attacker never modelled | `n = 1` |

The wrapper overrides `update` (the true dynamics) but leaves `phi_batch` (the attacker's
*model*, per-sample autograd through the MLP) linear-in-`a` on purpose — only our `α`/`B`
carry the assumption; BDQ and TRPO assumed nothing and are the control. `τ` is expressed as
a fraction `ρ` of the typical unclipped step norm, so the axis is scale-free.

**How to read it.** A clipped or multi-step victim is a genuinely *different control
problem*, not just a mismatched one — `null` moves too (hence the `cost/null` y-axis), and
the baselines are **not** guaranteed flat. So the quantity to read is the **gap between our
curve and the best baseline at each violation level**, not our own degradation against the
exact case, which confounds mismatch with a changed environment.


In [ ]:
class ClipVictim:
    """Clips the AGGREGATE update to norm tau -- breaks separability in `a`.

    phi_batch (the attacker's model) is left linear-in-`a` on purpose: the mismatch between
    `update` and `theta + a @ Phi` IS the experiment.
    """

    def __init__(self, base, tau):
        self.base, self.tau = base, tau
        self.eta, self.task = base.eta, getattr(base, "task", "mlp")

    def residual(self, theta, z):
        return self.base.residual(theta, z)

    def phi_batch(self, theta, z):
        return self.base.phi_batch(theta, z)

    def update(self, theta, z, a):
        step = a @ self.base.phi_batch(theta, z)
        nrm = step.norm()
        if self.tau is not None and float(nrm) > self.tau:
            step = step * (self.tau / nrm)
        return theta + step


class MultiStepVictim:
    """n SGD steps on the same reweighted batch -- breaks 'exactly one step'.

    n=1 is exactly the base victim, so the sweep has a clean exact anchor.
    """

    def __init__(self, base, n_steps):
        self.base, self.n_steps = base, n_steps
        self.eta, self.task = base.eta, getattr(base, "task", "mlp")

    def residual(self, theta, z):
        return self.base.residual(theta, z)

    def phi_batch(self, theta, z):
        return self.base.phi_batch(theta, z)     # attacker still assumes ONE step

    def update(self, theta, z, a):
        th = theta
        for _ in range(self.n_steps):
            th = th + a @ self.base.phi_batch(th, z)   # re-linearised at the new theta
        return th


def linearity_error(env, agent, n_states=60, seed=EVAL_SEED):
    """Mean ‖update(θ,z,a) − (θ + Σ aᵢφᵢ)‖ / ‖Σ aᵢφᵢ‖ over states the policy visits."""
    rng = torch.Generator().manual_seed(seed)
    errs, theta = [], env.theta_0.clone()
    for i in range(n_states):
        if i % env.T == 0:
            theta = env.theta_0.clone()
        z = env.stream.sample_batch(env.m, rng)
        a = agent.act(theta, z, i % env.T)
        pred = theta + a @ env.victim.phi_batch(theta, z)
        true = env.victim.update(theta, z, a).detach()
        errs.append(float((true - pred).norm() / ((pred - theta).norm() + 1e-12)))
        theta = true
    return float(np.mean(errs))


RHOS    = [None, 1.0, 0.6, 0.4, 0.25, 0.12]   # clip threshold / typical step norm
NSTEPS  = [1, 2, 3, 4]

if RUN_VIOLATION:
    # reference step norm: median ‖Σ aᵢφᵢ‖ under the unclipped greedy policy
    torch.manual_seed(SEEDS[0])
    _env0 = Environment(ECFG, replace(CFG, K_iters=K_COMPARE))
    _g0 = GreedyAgent(_env0.victim, _env0.theta_target, CFG, mode="sort")
    _rng, _norms, _th = torch.Generator().manual_seed(EVAL_SEED), [], _env0.theta_0.clone()
    for i in range(200):
        if i % _env0.T == 0:
            _th = _env0.theta_0.clone()
        _z = _env0.stream.sample_batch(_env0.m, _rng)
        _a = _g0.act(_th, _z, i % _env0.T)
        _norms.append(float((_a @ _env0.victim.phi_batch(_th, _z)).norm()))
        _th = _env0.victim.update(_th, _z, _a).detach()
    STEP0 = float(np.median(_norms))
    print(f"typical unclipped step norm = {STEP0:.4f}  -> tau = rho * {STEP0:.4f}\n")

    print("[aggregate gradient clipping]")
    res_clip = sweep(RHOS,
                     lambda r: (ECFG, replace(CFG, K_iters=K_COMPARE)),
                     seeds=(SEEDS[0],), label="rho",
                     post_env=lambda env, r: setattr(
                         env, "victim", ClipVictim(env.victim,
                                                   None if r is None else r * STEP0)))

    print("[multiple inner SGD steps]")
    res_step = sweep(NSTEPS,
                     lambda n: (ECFG, replace(CFG, K_iters=K_COMPARE)),
                     seeds=(SEEDS[0],), label="n_steps",
                     post_env=lambda env, n: setattr(
                         env, "victim", MultiStepVictim(env.victim, n)))

    # measured violation at each point (uses the greedy policy: no training needed)
    def violation_curve(values, wrap):
        out = []
        for v in values:
            torch.manual_seed(SEEDS[0])
            e = Environment(ECFG, replace(CFG, K_iters=K_COMPARE))
            e.victim = wrap(e.victim, v)
            out.append(linearity_error(
                e, GreedyAgent(e.victim, e.theta_target, CFG, mode="sort")) * 100)
        return out

    vio_clip = violation_curve(RHOS, lambda vic, r: ClipVictim(vic, None if r is None
                                                               else r * STEP0))
    vio_step = violation_curve(NSTEPS, MultiStepVictim)

    fig, (c1, c2) = plt.subplots(1, 2, figsize=(12.5, 4.3), sharey=True)
    for ax, xs, res_, xlab, ttl, inv in [
            (c1, range(len(RHOS)), res_clip,
             "clip threshold  τ / typical step", "Aggregate gradient clipping", True),
            (c2, NSTEPS, res_step,
             "inner SGD steps per round", "Multiple inner steps", False)]:
        nul = np.array(res_["null"]["cost"])
        for n in res_:
            ax.plot(list(xs), np.array(res_[n]["cost"]) / nul, f"{MARKERS[n]}-",
                    color=COLORS[n], lw=2, label=n)
        ax.axhline(1.0, color="0.7", ls=":", lw=1)
        ax.set_xlabel(xlab); ax.set_title(ttl); ax.legend(fontsize=8)
        if inv:
            ax.set_xticks(list(xs))
            ax.set_xticklabels(["exact" if r is None else f"{r:g}" for r in RHOS])
    c1.set_ylabel("cost / null")
    fig.suptitle("How much action-linearity violation does the advantage survive?", y=1.02)
    plt.tight_layout(); plt.show()

    other = [n for n in SWEEP_LEARNERS if n != "first-order"]
    print(f"\n{'setting':>22}{'violation':>12}{'first-order':>14}" +
          "".join(f"{n:>10}" for n in other))
    for lbl, res_, vio, vals in [("clip rho", res_clip, vio_clip, RHOS),
                                 ("inner steps", res_step, vio_step, NSTEPS)]:
        nul = np.array(res_["null"]["cost"])
        for i, v in enumerate(vals):
            tag = f"{lbl}={'exact' if v is None else v}"
            row = f"{tag:>22}{vio[i]:>11.1f}%{res_['first-order']['cost'][i]/nul[i]:>14.3f}"
            row += "".join(f"{res_[n]['cost'][i]/nul[i]:>10.3f}" for n in other)
            print(row)
    print("\nquote the violation column, not rho: it is the transferable number.")
else:
    print("skipped — set RUN_VIOLATION = True")


### 6e. Rank `r` — how much of the interaction `B` do you need?

§4 found `rank(B) ≤ min(m, D)` with `D=57 > m=10`, so unlike the `d<m` linear case — where
`m-d` eigenvalues of `B` vanish exactly and motivate `lowrank` structurally — `B` here is
generically **full rank**. There is no structural reason for a small `r` to suffice, so this
sweep is the direct test: does `low-rank(r)` need to climb all the way to `r=m` to match
`exact`, or does it still saturate early in practice?

**This sweep is free.** It reuses the `V̂` already trained in §2 and varies the selector at
decision time only, the same trick as `viz.cost_vs_dimension_tiers`. Nothing is retrained,
so cost differences here are *pure selection quality* on an identical value function.
`exact` (BQP / brute force) is the attainable floor; the duality gap is `low-rank`'s own
certificate against it.


In [ ]:
if RUN_RANK:
    from selection import select_action, lowrank_gap_trace

    RANKS = list(range(1, min(ECFG.m, env.d) + 1))

    def score_selector(cfg_sel):
        ag = MethodAgent(V, env.victim, env.theta_target, cfg_sel,
                         torch.Generator().manual_seed(1))
        c = evaluate(env, ag, torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
        return c, decision_latency_ms(env, ag)

    c_sort, l_sort = score_selector(replace(CFG, select_mode="sort"))
    c_exact, l_exact = score_selector(replace(CFG, select_mode="exact"))
    costs_r, lat_r = [], []
    for r in RANKS:
        c, l = score_selector(replace(CFG, select_mode="lowrank", rank_r=r))
        costs_r.append(c); lat_r.append(l)
        print(f"  r={r}: cost={c:.4f}  latency={l:.3f} ms", flush=True)

    # low-rank's own optimality certificate at each rank
    rng_r = torch.Generator().manual_seed(EVAL_SEED)
    gaps, theta_r = {r: [] for r in RANKS}, env.theta_0.clone()
    for i in range(60):
        if i % env.T == 0:
            theta_r = env.theta_0.clone()
        z = env.stream.sample_batch(env.m, rng_r)
        alpha, B = scores_and_interaction(V, theta_r, z, i % env.T, env.victim,
                                          env.theta_target, CFG.lam, need_B=True,
                                          use_hessian=CFG.use_hessian_in_B)
        for r in RANKS:
            gaps[r].append(lowrank_gap_trace(alpha, B, c_edit=CFG.c_edit,
                                             k_max=CFG.k_max, balance=CFG.balance,
                                             r=r, dual_iters=CFG.dual_iters)[-1])
        a_r = select_action(alpha, None, "sort", c_edit=CFG.c_edit,
                            balance=CFG.balance, k_max=CFG.k_max)
        theta_r = env.victim.update(theta_r, z, a_r).detach()

    fig, (q1, q2, q3) = plt.subplots(1, 3, figsize=(16, 4.3))
    q1.plot(RANKS, costs_r, "o-", color=COLORS["low-rank"], lw=2, label="low-rank(r)")
    q1.axhline(c_sort, color=COLORS["first-order"], ls="--", lw=2,
               label=f"first-order ({c_sort:.3f})")
    q1.axhline(c_exact, color=COLORS["exact"], ls="-.", lw=2, label=f"exact ({c_exact:.3f})")
    q1.set_xlabel("rank r"); q1.set_ylabel("mean episode cost")
    q1.set_title(f"Selection quality (shared V̂, rank(B) ≤ {min(ECFG.m, env.d)})")
    q1.set_xticks(RANKS); q1.legend(fontsize=8)

    q2.plot(RANKS, lat_r, "o-", color=COLORS["low-rank"], lw=2, label="low-rank(r)")
    q2.axhline(l_sort, color=COLORS["first-order"], ls="--", lw=2, label="first-order")
    q2.axhline(l_exact, color=COLORS["exact"], ls="-.", lw=2, label="exact")
    q2.set_xlabel("rank r"); q2.set_ylabel("ms per action"); q2.set_yscale("log")
    q2.set_title("Price of the rank"); q2.set_xticks(RANKS); q2.legend(fontsize=8)

    mg = [float(np.mean(gaps[r])) for r in RANKS]
    neg = [int(np.sum(np.array(gaps[r]) < 0)) for r in RANKS]
    q3.plot(RANKS, mg, "o-", color=COLORS["low-rank"], lw=2)
    q3.axhline(0, color="k", lw=0.8)
    q3.set_xlabel("rank r"); q3.set_ylabel("mean final duality gap")
    q3.set_title("Certificate quality (negative = bound invalid)")
    q3.set_xticks(RANKS)
    plt.tight_layout(); plt.show()

    print(f"\nfirst-order {c_sort:.4f}   exact {c_exact:.4f}   "
          f"headroom {(c_sort-c_exact)/c_sort*100:+.2f}%")
    print(f"best low-rank: r={RANKS[int(np.argmin(costs_r))]} "
          f"({min(costs_r):.4f}, {(c_sort-min(costs_r))/c_sort*100:+.2f}% vs first-order)")
    print("negative gaps by rank:", {r: f"{n}/60" for r, n in zip(RANKS, neg)})
else:
    print("skipped — set RUN_RANK = True")


### Takeaways

1. **Action-linearity is a property of the *update rule*, not the model** (§1 verifies it to
   ~1e-8 on the MLP). The decomposition therefore applies to an arbitrary differentiable
   victim; only `phi_batch` gets more expensive, needing per-sample gradients.
2. **`θ` moves to parameter space.** `env.d` becomes the parameter count `D=57`, and since
   `D > m`, `B` is generically full rank — the low-rank structure that `d < m` gave us in
   the linear case is gone, which is exactly where `lowrank` should start to matter.
3. **The victim's own stability now binds.** Above `eta≈0.5` the MLP diverges and no attack
   is measurable — a constraint linear victims never impose.
4. **Myopia becomes actively harmful.** At `eta=0.5` greedy is *worse than null* (it
   destabilises the victim) while the planning method still reduces cost. This is the
   cleanest evidence yet that cost-to-go matters, and it only appears on a nonlinear victim.

**Caveats.** `θ̃` is a random point in weight space; because of weight-space symmetries
(permuting hidden units leaves `f` unchanged) it is a well-defined but somewhat arbitrary
target — an equally valid formulation would target the victim's *function* rather than its
*weights*. Results are single-seed; re-run with several before citing numbers.